# 🍟 Snackspert - Website Recensies naar Google Docs

Dit notebook verzamelt alle recensies van **snackspert.nl/restaurant** en maakt per recensie een apart Google Docs-bestand aan in een Google Drive-map.

## Hoe te gebruiken
1. Voer elke cel uit door op het **▶ play-knopje** te klikken (of druk `Shift+Enter`)
2. Bij stap 2 moet je inloggen met je Google-account
3. Stap 3 t/m 5 doen het werk!

---

## Stap 1: Installeer benodigde packages
Dit hoef je maar één keer te doen per sessie.

In [ ]:
!pip install -q requests beautifulsoup4 google-api-python-client google-auth-httplib2 google-auth-oauthlib
print("\n\u2705 Alle packages ge\u00efnstalleerd!")

## Stap 2: Log in met je Google-account
Er verschijnt een pop-up om in te loggen. Dit geeft het notebook toegang tot je Google Drive en Google Docs.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.auth import default
creds, _ = default()

from googleapiclient.discovery import build
docs_service = build('docs', 'v1', credentials=creds)
drive_service = build('drive', 'v3', credentials=creds)

print("\u2705 Ingelogd en verbonden met Google Drive & Docs!")

## Stap 3: Instellingen

In [ ]:
# === INSTELLINGEN ===

# Naam van de Google Drive-map waar de documenten in komen
# (wordt automatisch aangemaakt als deze niet bestaat)
DRIVE_MAP_NAAM = "Snackspert Recensies"

# Maximaal aantal recensies ophalen (0 = alles)
MAX_RECENSIES = 0

print(f"\u2705 Instellingen geladen:")
print(f"   Drive-map: {DRIVE_MAP_NAAM}")
print(f"   Max recensies: {'alles' if MAX_RECENSIES == 0 else MAX_RECENSIES}")

## Stap 4: Recensies ophalen van snackspert.nl

Dit haalt alle restaurants op via de WordPress REST API en scrapet vervolgens elke restaurantpagina voor de recensietekst en sterren.

**Dit kan 10-30 minuten duren** afhankelijk van het aantal restaurants (~700).

In [ ]:
import re
import time
import requests
from dataclasses import dataclass
from bs4 import BeautifulSoup


@dataclass
class Recensie:
    """Een enkele Snackspert-recensie."""
    naam: str
    adres: str
    tekst: str
    sterren: float
    sterren_tekst: str
    afbeelding_url: str
    pagina_url: str


def tel_sterren(tekst: str) -> tuple:
    """Tel het aantal sterren (emoji's) in de recensietekst."""
    # Tel volle sterren
    volle = tekst.count('\u2b50') + tekst.count('\u2b50\ufe0f')
    # \u2b50\ufe0f en \u2b50 kunnen overlappen, tel uniek
    volle = len(re.findall(r'\u2b50\ufe0f?', tekst))
    # Tel halve sterren
    halve = tekst.count('\u00bd') + tekst.count('1/2')
    totaal = volle + (0.5 if halve > 0 else 0)
    # Maak een leesbare tekst
    sterren_str = '\u2b50' * volle + ('\u00bd' if halve > 0 else '')
    return totaal, sterren_str


def scrape_restaurant_pagina(url: str, sessie: requests.Session) -> dict:
    """Scrape een individuele restaurantpagina."""
    resp = sessie.get(url, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')

    # Naam
    naam_el = soup.select_one('.restaurantBlock .bigTitle')
    naam = naam_el.get_text(strip=True) if naam_el else ''

    # Adres
    adres_el = soup.select_one('.restaurantBlock .innerAddress')
    adres = adres_el.get_text(strip=True) if adres_el else ''

    # Recensietekst
    tekst_el = soup.select_one('.restaurantBlock .text')
    tekst = tekst_el.get_text(strip=True) if tekst_el else ''

    # Sterren uit de tekst halen
    sterren, sterren_tekst = tel_sterren(tekst)

    # Afbeelding
    img_el = soup.select_one('.restaurantBlock .innerImage')
    afbeelding_url = ''
    if img_el and img_el.get('style'):
        match = re.search(r"url\('([^']+)'\)", img_el['style'])
        if match:
            afbeelding_url = match.group(1)

    return {
        'naam': naam,
        'adres': adres,
        'tekst': tekst,
        'sterren': sterren,
        'sterren_tekst': sterren_tekst,
        'afbeelding_url': afbeelding_url,
    }


# --- Stap 1: Alle restaurant-URLs ophalen via de WP REST API ---
print("\U0001f50d Stap 4a: Alle restaurants ophalen via de REST API...\n")

sessie = requests.Session()
sessie.headers.update({
    'User-Agent': 'Mozilla/5.0 (X11; CrOS x86_64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36'
})

alle_restaurants = []
pagina = 1
while True:
    api_url = f"https://snackspert.nl/wp-json/wp/v2/restaurant?per_page=100&page={pagina}"
    resp = sessie.get(api_url, timeout=30)
    if resp.status_code != 200:
        break
    data = resp.json()
    if not data:
        break
    for item in data:
        alle_restaurants.append({
            'id': item['id'],
            'naam': item['title']['rendered'],
            'url': item['link'],
        })
    totaal = int(resp.headers.get('X-WP-Total', 0))
    totaal_paginas = int(resp.headers.get('X-WP-TotalPages', 0))
    print(f"  Pagina {pagina}/{totaal_paginas} - {len(alle_restaurants)}/{totaal} restaurants")
    if pagina >= totaal_paginas:
        break
    pagina += 1
    time.sleep(0.5)  # Netjes wachten

print(f"\n\u2705 {len(alle_restaurants)} restaurants gevonden!\n")

# --- Stap 2: Elke restaurantpagina scrapen ---
print("\U0001f50d Stap 4b: Elke restaurantpagina scrapen voor recensie + sterren...\n")

if MAX_RECENSIES > 0:
    alle_restaurants = alle_restaurants[:MAX_RECENSIES]
    print(f"   (Beperkt tot {MAX_RECENSIES} recensies)\n")

recensies = []
fouten = []
for i, rest in enumerate(alle_restaurants, 1):
    print(f"  [{i}/{len(alle_restaurants)}] {rest['naam'][:50]}...", end=" ")
    try:
        data = scrape_restaurant_pagina(rest['url'], sessie)
        recensie = Recensie(
            naam=data['naam'] or rest['naam'],
            adres=data['adres'],
            tekst=data['tekst'],
            sterren=data['sterren'],
            sterren_tekst=data['sterren_tekst'],
            afbeelding_url=data['afbeelding_url'],
            pagina_url=rest['url'],
        )
        recensies.append(recensie)
        print(f"\u2705 {data['sterren']} sterren")
    except Exception as e:
        fouten.append({'naam': rest['naam'], 'url': rest['url'], 'fout': str(e)})
        print(f"\u274c {e}")
    # Wacht kort om de server niet te overbelasten
    if i % 10 == 0:
        time.sleep(1)
    else:
        time.sleep(0.3)

print(f"\n\u2705 {len(recensies)} recensies opgehaald!")
if fouten:
    print(f"\u26a0\ufe0f {len(fouten)} fouten opgetreden")

## Stap 5: Recensies opslaan als Google Docs
Per recensie wordt een apart document aangemaakt in de map op je Google Drive.

In [ ]:
import html as html_module

# --- Google Drive-map zoeken of aanmaken ---
print(f"\U0001f4c1 Map '{DRIVE_MAP_NAAM}' zoeken of aanmaken...\n")

query = f"name = '{DRIVE_MAP_NAAM}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
result = drive_service.files().list(q=query, fields='files(id, name)').execute()
folders = result.get('files', [])

if folders:
    folder_id = folders[0]['id']
    print(f"\u2705 Bestaande map gevonden: {DRIVE_MAP_NAAM}")
else:
    folder_metadata = {
        'name': DRIVE_MAP_NAAM,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    folder = drive_service.files().create(body=folder_metadata, fields='id').execute()
    folder_id = folder['id']
    print(f"\u2705 Nieuwe map aangemaakt: {DRIVE_MAP_NAAM}")

print(f"   Map-ID: {folder_id}")
print(f"   \U0001f517 https://drive.google.com/drive/folders/{folder_id}\n")


def maak_recensie_doc(recensie, folder_id):
    """Maak een Google Docs-bestand aan voor \u00e9\u00e9n recensie."""
    clean_naam = html_module.unescape(recensie.naam)
    doc_title = f"Snackspert - {clean_naam}"

    # Leeg document aanmaken
    doc = docs_service.documents().create(body={'title': doc_title}).execute()
    doc_id = doc['documentId']

    # Verplaats naar de juiste map
    drive_service.files().update(
        fileId=doc_id,
        addParents=folder_id,
        removeParents='root',
        fields='id, parents'
    ).execute()

    # Document vullen met inhoud
    sections = []

    # Titel
    sections.append({'text': f"{clean_naam}\n", 'style': 'HEADING_1'})

    # Sterren
    sterren_display = f"{recensie.sterren} van 5 sterren"
    if recensie.sterren_tekst:
        sterren_display = f"{recensie.sterren_tekst} ({recensie.sterren}/5)"
    sections.append({'text': f"Beoordeling: {sterren_display}\n\n", 'style': 'NORMAL_TEXT'})

    # Metadata
    meta_lines = []
    if recensie.adres:
        meta_lines.append(f"Adres: {recensie.adres}")
    meta_lines.append(f"Website: {recensie.pagina_url}")
    sections.append({'text': '\n'.join(meta_lines) + '\n\n', 'style': 'NORMAL_TEXT'})

    # Recensie tekst
    sections.append({'text': 'Recensie\n', 'style': 'HEADING_2'})
    sections.append({'text': (recensie.tekst or '(Geen tekst)') + '\n\n', 'style': 'NORMAL_TEXT'})

    # Afbeelding link
    if recensie.afbeelding_url:
        sections.append({'text': 'Afbeelding\n', 'style': 'HEADING_2'})
        sections.append({'text': recensie.afbeelding_url + '\n', 'style': 'NORMAL_TEXT'})

    # Requests opbouwen
    requests_list = []
    index = 1
    for section in sections:
        text = section['text']
        requests_list.append({
            'insertText': {
                'location': {'index': index},
                'text': text,
            }
        })
        if section['style'] != 'NORMAL_TEXT':
            requests_list.append({
                'updateParagraphStyle': {
                    'range': {'startIndex': index, 'endIndex': index + len(text)},
                    'paragraphStyle': {'namedStyleType': section['style']},
                    'fields': 'namedStyleType',
                }
            })
        index += len(text)

    if requests_list:
        docs_service.documents().batchUpdate(
            documentId=doc_id, body={'requests': requests_list}
        ).execute()

    return f"https://docs.google.com/document/d/{doc_id}/edit"


# --- Alle recensies verwerken ---
print(f"\U0001f4dd {len(recensies)} documenten aanmaken...\n")

resultaten = []
for i, recensie in enumerate(recensies, 1):
    clean_naam = html_module.unescape(recensie.naam)
    print(f"  [{i}/{len(recensies)}] {clean_naam[:50]}...", end=" ")
    try:
        url = maak_recensie_doc(recensie, folder_id)
        resultaten.append({'naam': clean_naam, 'sterren': recensie.sterren, 'url': url})
        print(f"\u2705")
    except Exception as e:
        resultaten.append({'naam': clean_naam, 'sterren': recensie.sterren, 'url': None, 'error': str(e)})
        print(f"\u274c {e}")

# Samenvatting
gelukt = [r for r in resultaten if r.get('url')]
mislukt = [r for r in resultaten if not r.get('url')]

print(f"\n{'='*50}")
print(f"\u2705 KLAAR!")
print(f"{'='*50}")
print(f"  Totaal:    {len(recensies)} recensies")
print(f"  Gelukt:    {len(gelukt)}")
if mislukt:
    print(f"  Mislukt:   {len(mislukt)}")
print(f"\n\U0001f4c2 Open je Google Drive-map:")
print(f"   https://drive.google.com/drive/folders/{folder_id}")

## Stap 6 (optioneel): Bekijk een overzicht
Bekijk een tabel met alle aangemaakte documenten, gesorteerd op sterren.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        'Restaurant': r['naam'][:40],
        'Sterren': r['sterren'],
        'Status': '\u2705' if r.get('url') else '\u274c',
        'Google Docs Link': r.get('url', r.get('error', '-'))
    }
    for r in resultaten
])

df_sorted = df.sort_values('Sterren', ascending=False)

print(f"\U0001f4ca Overzicht van alle {len(resultaten)} recensies:\n")
print(f"Gemiddelde score: {df['Sterren'].mean():.1f} sterren")
print(f"Hoogste score:    {df['Sterren'].max()} sterren")
print(f"Laagste score:    {df['Sterren'].min()} sterren")
print()
df_sorted